## A. Memuat Data dan Preprocessing

Sel ini menyiapkan data teks sebelum dipakai untuk melatih model.

**Langkah-langkahnya:**
1. **Memuat data** dari `data_berita_sport_finance_saya.csv` (kolom `Kategori`, `Judul Berita`, `Isi Berita`).
2. **Menggabungkan** judul dan isi berita menjadi satu kolom `teks`.
3. **Membuat label** dari kolom `Kategori` (huruf besar: `SPORT` atau `FINANCE`).
4. **Preprocessing** dibuat dalam dua versi, karena tiap metode butuh perlakuan berbeda:

| Versi | Dipakai untuk | Tahapan |
|---|---|---|
| `teks_sg` | Skip-gram | cleansing, case folding, tokenisasi |
| `teks_tf` | TF-IDF | cleansing, case folding, tokenisasi, **stopword removal**, **stemming** (Sastrawi) |

**Penjelasan istilah:**
- **Cleansing**: membuang angka, tanda baca, dan simbol, hanya menyisakan huruf.
- **Case folding**: mengubah semua huruf menjadi huruf kecil.
- **Tokenisasi**: memecah kalimat menjadi kata-kata.
- **Stopword removal**: membuang kata umum yang tidak membawa makna (misalnya *yang*, *dan*, *untuk*).
- **Stemming**: mengubah kata ke bentuk dasar (misalnya *berlari* menjadi *lari*).

> **Penting:** fungsi `bersihkan_teks` di sini harus **sama persis** dengan yang ada di `app.py`. Jika berbeda, hasil prediksi di aplikasi akan tidak akurat.
>
> Sel ini bisa berjalan agak lama karena proses stemming Sastrawi.

In [2]:
import re, joblib, numpy as np, pandas as pd, nltk
from functools import lru_cache
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("stopwords", quiet=True)
stop = set(stopwords.words("indonesian")).union(set(stopwords.words("english")))
stop |= {
    'dan','atau','yang','dengan','juga','karena','namun','sehingga','adalah','akan',
    'saat','pada','untuk','dari','oleh','dalam','dapat','kami','mereka','kita','saya',
    'ini','itu','tersebut','tsb','dll','dkk','spt','yg','dgn','dr','jr','sr','thn',
    'tdk','gak','nggak','ya','ia','terkait','berdasarkan','setelah','sebelum',
    'sambil','selain','sekitar','hingga','maupun','bisa','per','waktu','apakah','adanya'
}
stemmer = StemmerFactory().create_stemmer()

@lru_cache(maxsize=None)
def stem_cached(w):
    return stemmer.stem(w)

def bersihkan_teks(text, hapus_stopword=True, stemming=True):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if len(w) > 1]
    if hapus_stopword:
        words = [w for w in words if w not in stop]
    if stemming:
        words = [stem_cached(w) for w in words]
    return " ".join(words)
df = pd.read_csv("data_berita_sport_finance_saya.csv")
df.columns = df.columns.str.strip()   # jaga-jaga ada spasi di ujung nama kolom

df["teks"] = df["Judul Berita"].fillna("") + " " + df["Isi Berita"].fillna("")
df["label"] = df["Kategori"].astype(str).str.strip().str.upper()

print(df["label"].value_counts())     # harus muncul SPORT dan FINANCE

df["teks_sg"] = df["teks"].apply(lambda t: bersihkan_teks(t, False, False))  # untuk Skip-gram
df["teks_tf"] = df["teks"].apply(lambda t: bersihkan_teks(t, True, True))    # untuk TF-IDF

label
SPORT      100
FINANCE     96
Name: count, dtype: int64


## B. Model Skip-gram + Gaussian Naive Bayes

Sel ini melatih model klasifikasi berbasis **Word2Vec (Skip-gram)**.

**Cara kerjanya:**
1. **Membagi data** menjadi 80% data latih dan 20% data uji (`train_test_split`, dengan `stratify` agar proporsi SPORT/FINANCE tetap seimbang).
2. **Melatih Word2Vec Skip-gram** (`sg=1`) agar tiap kata punya vektor 100 dimensi yang menangkap makna dan konteksnya.
3. **Membuat vektor dokumen**: vektor semua kata dalam satu berita **dirata-rata** menjadi satu vektor (fungsi `doc_vector`).
4. **Melatih GaussianNB** pada vektor dokumen tersebut. GaussianNB dipilih karena inputnya berupa angka kontinu (bukan hitungan kata).
5. **Evaluasi** dengan `classification_report` (precision, recall, f1-score) pada data uji.
6. **Model final**: Word2Vec dan GaussianNB dilatih ulang memakai **seluruh data**, lalu disimpan.

**Parameter Word2Vec:**
- `vector_size=100`: panjang vektor tiap kata
- `window=5`: jumlah kata di kiri dan kanan yang dianggap konteks
- `min_count=2`: kata yang muncul kurang dari 2 kali diabaikan
- `sg=1`: memakai arsitektur Skip-gram

**Output:** file `model_skipgram_nb.pkl` berisi vektor kata (`wv`) dan classifier (`clf`), disimpan di folder `klasifikasi_app`.

In [3]:
from gensim.models import Word2Vec
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def doc_vector(tokens, wv):
    vecs = [wv[t] for t in tokens if t in wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)

tr, te = train_test_split(df.index, test_size=0.2, stratify=df["label"], random_state=42)

# evaluasi
sg = Word2Vec([t.split() for t in df.loc[tr, "teks_sg"]],
              vector_size=100, window=5, min_count=2, sg=1, workers=4, seed=42)
X_tr = np.vstack([doc_vector(t.split(), sg.wv) for t in df.loc[tr, "teks_sg"]])
X_te = np.vstack([doc_vector(t.split(), sg.wv) for t in df.loc[te, "teks_sg"]])
print(classification_report(df.loc[te, "label"], GaussianNB().fit(X_tr, df.loc[tr, "label"]).predict(X_te)))

# model final (seluruh data)
model_sg = Word2Vec([t.split() for t in df["teks_sg"]],
                    vector_size=100, window=5, min_count=2, sg=1, workers=4, seed=42)
X_all = np.vstack([doc_vector(t.split(), model_sg.wv) for t in df["teks_sg"]])
gnb_final = GaussianNB().fit(X_all, df["label"])

joblib.dump({"wv": model_sg.wv, "clf": gnb_final}, "../klasifikasi_app/model_skipgram_nb.pkl")
print("Skip-gram tersimpan")

              precision    recall  f1-score   support

     FINANCE       1.00      0.55      0.71        20
       SPORT       0.69      1.00      0.82        20

    accuracy                           0.78        40
   macro avg       0.84      0.78      0.76        40
weighted avg       0.84      0.78      0.76        40

Skip-gram tersimpan


## C. Model TF-IDF + Multinomial Naive Bayes

Sel ini melatih model klasifikasi berbasis **bobot kata TF-IDF**.

**Cara kerjanya:**
1. **TF-IDF** (`TfidfVectorizer`) mengubah tiap berita menjadi vektor angka. Kata yang sering muncul di satu berita tetapi jarang di berita lain mendapat bobot tinggi, karena dianggap lebih khas.
2. **Evaluasi** dilakukan dengan data latih dan uji yang sama seperti di Sel B, supaya perbandingan kedua metode adil.
3. **MultinomialNB** dipakai sebagai classifier karena cocok untuk data berbasis frekuensi atau bobot kata.
4. **Model final**: vectorizer dan classifier dilatih ulang memakai **seluruh data**, lalu disimpan.

**Rumus singkat TF-IDF:**
`TF-IDF = TF × IDF`
- **TF**: seberapa sering kata muncul dalam satu dokumen
- **IDF**: seberapa jarang kata muncul di seluruh dokumen

**Output:** file `model_tfidf_nb.pkl` berisi `vectorizer` dan `clf`, disimpan di folder `klasifikasi_app`.

---
**Perbandingan kedua metode**

| | Skip-gram + GaussianNB | TF-IDF + MultinomialNB |
|---|---|---|
| Representasi | Rata-rata vektor kata (padat, 100 dimensi) | Bobot kata (jarang/sparse, sesuai ukuran kosakata) |
| Menangkap makna kata | Ya (konteks) | Tidak (hanya frekuensi) |
| Stopword removal & stemming | Tidak | Ya |
| Classifier | GaussianNB | MultinomialNB |

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

tfidf_eval = TfidfVectorizer()
Xt = tfidf_eval.fit_transform(df.loc[tr, "teks_tf"])
print(classification_report(df.loc[te, "label"],
      MultinomialNB().fit(Xt, df.loc[tr, "label"]).predict(tfidf_eval.transform(df.loc[te, "teks_tf"]))))

tfidf_final = TfidfVectorizer()
X_all_tf = tfidf_final.fit_transform(df["teks_tf"])
mnb_final = MultinomialNB().fit(X_all_tf, df["label"])

joblib.dump({"vectorizer": tfidf_final, "clf": mnb_final}, "../klasifikasi_app/model_tfidf_nb.pkl")
print("TF-IDF tersimpan")

              precision    recall  f1-score   support

     FINANCE       1.00      1.00      1.00        20
       SPORT       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

TF-IDF tersimpan


: 